In [49]:
# Imports 

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, balanced_accuracy_score, confusion_matrix
from imblearn.over_sampling import SMOTE


Pipeline and ColumnTransformer -> Scaling preprocessing assistants, (SciKit-Learn Website)

Chosen Metrics; Imbalance robustness, PR_AUC, balanced accuracy score, commonly recomended for fraud detection

Sources; 

https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

https://scikit-learn.org/stable/modules/model_evaluation.html
    
https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html

https://scikit-learn.org/stable/modules/compose.html
            

In [17]:
path_cc = path = "D:/Final Year/App Domains/Project/datasets/creditcard.csv"
path_synfin = path = "D:/Final Year/App Domains/Project/datasets/Synthetic_Financial_Dataset.csv"
 
cc = pd.read_csv(path_cc)
temp_df = pd.read_csv(path_synfin)

Define Key column names, generates a Temporal train/split function with a feature slection helper. 

Fraud models are trained on past and tested on future hence why the leakage prevention is so important 

Exclude target feature and retain rest for clean pipeline ML. 

Sources; 

https://fraud-detection-handbook.github.io/fraud-detection-handbook/Chapter_6_ImbalancedLearning/Introduction.html

https://fraud-detection-handbook.github.io/fraud-detection-handbook/Chapter_6_ImbalancedLearning/Resampling.html

https://arxiv.org/html/2506.02703v1

In [18]:
time_col   = "Time"  
class_col = "Class"  

# Have to geenrate a train/split function rather than using scikit_learns train_test_split()
# Copied from Q1


def temporal_train_test_split(
    df: pd.DataFrame,
    time_col: int,
    class_col: str,
    train_frac: float = 0.8):
    
    # Sort df by time_col, do a 80/20 temporal split.
    # Returns: X_train, y_train, X_test, y_test
    
    # Sort by time ascending
    df_sorted = df.sort_values(by=time_col).reset_index(drop=True)

    # Compute split index
    split_idx = int(len(df_sorted) * train_frac)

    # Split
    train_df = df_sorted.iloc[:split_idx].copy()
    test_df  = df_sorted.iloc[split_idx:].copy()

    # Separate features and target
    X_train = train_df.drop(columns=[class_col])
    y_train = train_df[class_col].copy()

    X_test  = test_df.drop(columns=[class_col])
    y_test  = test_df[class_col].copy()

    return X_train, y_train, X_test, y_test, df_sorted

def feature_columns(df: pd.DataFrame, class_col: str):
    
    #Return numeric feature columns
    
    return [c for c in df.columns if c != class_col]

Below is an alternative Pandas slicing test/trial for the train/split function. 

Testing out time index intself directly rather than row count. 

Yielded same values when tested against the above. 

In [19]:
def temporal_split(df, time_col='Time', target='Class', frac=0.8):
    """Split by time quantile instead of index count."""
    cutoff = df[time_col].quantile(frac)
    train_df = df[df[time_col] <= cutoff].copy()
    test_df  = df[df[time_col] > cutoff].copy()

    X_train, y_train = train_df.drop(columns=[target]), train_df[target].astype(int)
    X_test,  y_test  = test_df.drop(columns=[target]), test_df[target].astype(int)

    return X_train, y_train, X_test, y_test


Define Baseline models in order to build identical hyperparameter baselines. 

    * Logistic regression, scaled through ColumnTransformer inside a pipeline 
    * Random Forest 
    
Pipeline secures prepprocessing and modelling with the aid of leakage avoidance and reproducibility improvement 
Scaling utlised for ...

Sources; 

https://scikit-learn.org/stable/modules/compose.html

https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html

https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

In [20]:
def make_baseline_models(numeric_features):
    
    # Returns two sklearn Pipelines: Logistic Regression and Random Forest
    # with simple, fixed hyperparameters. No class_weight yet (true baseline).
    
    # Preprocess numeric features. Scale for LR. RF does not require scaling.
    
    scaler = ColumnTransformer(
        transformers=[("num", StandardScaler(with_mean=True, with_std=True), numeric_features)],
        remainder="drop"
    )

    # LR Base
    lr = Pipeline(steps=[
        ("scale", scaler),
        ("clf", LogisticRegression(
            solver="liblinear",      # robust for small/medium data
            C=1.0,                   # fixed baseline
            max_iter=200,
            class_weight=None,       # <-- keep None for baseline
            random_state=29
        ))
    ])

    # RF Base
    rf = Pipeline(steps=[
        ("clf", RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            min_samples_leaf=1,
            n_jobs=-1,
            class_weight=None,       # <-- keep None for baseline
            random_state=29
        ))
    ])

    return {
        "LR_baseline": lr,
        "RF_baseline": rf
    }


Begin tailored evaluation, fitting models, compute ROC-AUC, PR-AUC, F1 score, Balanced Acc and a Confusion Matrix 

Average Precision (PR-AUC) is the most utlised measure under heavy imbalance as it presents a better trade off of precion/recall than ROC due to the skewness. 

Sources; 

https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html

https://scikit-learn.org/stable/modules/model_evaluation.html

In [21]:
def evaluate_baselines(models: dict, X_train, y_train, X_test, y_test):
    results = []

    for name, pipe in models.items():
        # Fit on training only
        pipe.fit(X_train, y_train)

        # Probabilities for AUC metrics
        if hasattr(pipe[-1], "predict_proba"):
            y_proba = pipe.predict_proba(X_test)[:, 1]
        elif hasattr(pipe[-1], "decision_function"):
            from sklearn.preprocessing import MinMaxScaler
            scores = pipe.decision_function(X_test).reshape(-1, 1)
            y_proba = MinMaxScaler().fit_transform(scores).ravel()
        else:
            y_proba = pipe.predict(X_test)

        y_pred = pipe.predict(X_test)

        roc  = roc_auc_score(y_test, y_proba)
        pr   = average_precision_score(y_test, y_proba)  # PR-AUC
        f1   = f1_score(y_test, y_pred, zero_division=0)
        bacc = balanced_accuracy_score(y_test, y_pred)
        cm   = confusion_matrix(y_test, y_pred)

        results.append({
            "model": name,
            "roc_auc": roc,
            "pr_auc": pr,
            "f1": f1,
            "balanced_accuracy": bacc,
            "tn": cm[0, 0], "fp": cm[0, 1],
            "fn": cm[1, 0], "tp": cm[1, 1],
        })

    return pd.DataFrame(results).sort_values(by=["pr_auc", "roc_auc"], ascending=False).reset_index(drop=True)


In [37]:
def evaluate_models(models: dict, X_train, y_train, X_test, y_test):
    rows = []
    for name, pipe in models.items():
        pipe.fit(X_train, y_train)
        # probabilities (AUC metrics)
        if hasattr(pipe[-1], "predict_proba"):
            y_proba = pipe.predict_proba(X_test)[:,1]
        elif hasattr(pipe[-1], "decision_function"):
            scores = pipe.decision_function(X_test)
            # normalize to [0,1] for AUC if needed
            y_proba = (scores - scores.min())/(scores.max()-scores.min() + 1e-12)
        else:
            y_proba = pipe.predict(X_test)

        y_pred = pipe.predict(X_test)

        rows.append({
            "model": name,
            "roc_auc": roc_auc_score(y_test, y_proba),
            "pr_auc":  average_precision_score(y_test, y_proba),  # PR-AUC
            "f1":      f1_score(y_test, y_pred, zero_division=0),
            "bacc":    balanced_accuracy_score(y_test, y_pred),
            **dict(zip(["tn","fp","fn","tp"], confusion_matrix(y_test, y_pred).ravel()))
        })
    return pd.DataFrame(rows).sort_values(["pr_auc","roc_auc"], ascending=False).reset_index(drop=True)


Executes a baseline without weighting, to compare aginst imbalance handling methods. 

Sorted by time and  optional train/test distribution for optimization. 

Sources; 

https://fraud-detection-handbook.github.io/fraud-detection-handbook/Chapter_6_ImbalancedLearning/Introduction.html

https://scikit-learn.org/stable/modules/compose.html

https://fraud-detection-handbook.github.io/fraud-detection-handbook/Chapter_6_ImbalancedLearning/Resampling.html

https://arxiv.org/html/2506.02703v1

In [43]:
# Make the split
X_train, y_train, X_test, y_test = temporal_train_test_split(cc, time_col, class_col, train_frac=0.70)

# Define feature set 
num_feats = feature_columns(cc, class_col)

# Build and evaluate baselines
models = make_baseline_models(num_feats)
baseline_report = evaluate_baselines(models, X_train, y_train, X_test, y_test)

baseline_report


,model,roc_auc,pr_auc,f1,balanced_accuracy,tn,fp,fn,tp
0,RF_baseline,0.953929,0.805246,0.802198,0.837957,85334,1,35,73
1,LR_baseline,0.971374,0.710327,0.625000,0.754553,85322,13,53,55


Implementing an 80/20 split over a 70/30, what happens ? 

In [44]:
# Make the split
X_train, y_train, X_test, y_test = temporal_train_test_split(cc, time_col, class_col, train_frac=0.80)

# Define feature set 
num_feats = feature_columns(cc, class_col)

# Build and evaluate baselines
models = make_baseline_models(num_feats)
baseline_report = evaluate_baselines(models, X_train, y_train, X_test, y_test)

baseline_report

,model,roc_auc,pr_auc,f1,balanced_accuracy,tn,fp,fn,tp
0,RF_baseline,0.962488,0.791951,0.812500,0.846658,56886,1,23,52
1,LR_baseline,0.975349,0.739719,0.694215,0.779965,56883,4,33,42


### Begin comparing SMOTE, Undersampling and Anomaly-Based Methodology. 

Sources; 

https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTE.html
    
https://imbalanced-learn.org/stable/

In [45]:
# SMOTE 


random_state=29

def smote_variants(numeric_features):
    scaler = ColumnTransformer([("num", StandardScaler(), numeric_features)], remainder="drop")
    lr = Pipeline([("scale", scaler),
                   ("clf", LogisticRegression(solver="liblinear", C=1.0, max_iter=200,
                                              class_weight=None, random_state=random_state))])
    rf = Pipeline([("clf", RandomForestClassifier(n_estimators=200, n_jobs=-1,
                                                  class_weight=None, random_state=random_state))])
    return {"LR_SMOTE": ("smote", lr), "RF_SMOTE": ("smote", rf)}

def fit_eval_with_smote(model_dict, X_train, y_train, X_test, y_test):
    rows = []
    smote = SMOTE(random_state=random_state)  # see docs for k_neighbors/sampling_strategy
    for name, (tag, pipe) in model_dict.items():
        X_tr_rs, y_tr_rs = smote.fit_resample(X_train, y_train)  # TRAIN ONLY
        pipe.fit(X_tr_rs, y_tr_rs)
        y_proba = pipe.predict_proba(X_test)[:,1] if hasattr(pipe[-1], "predict_proba") else pipe.predict(X_test)
        y_pred  = pipe.predict(X_test)
        rows.append({
            "model": name,
            "roc_auc": roc_auc_score(y_test, y_proba),
            "pr_auc":  average_precision_score(y_test, y_proba),
            "f1":      f1_score(y_test, y_pred, zero_division=0),
            "bacc":    balanced_accuracy_score(y_test, y_pred),
            **dict(zip(["tn","fp","fn","tp"], confusion_matrix(y_test, y_pred).ravel()))
        })
    return pd.DataFrame(rows).sort_values(["pr_auc","roc_auc"], ascending=False).reset_index(drop=True)


Sources; 

https://imbalanced-learn.org/stable/under_sampling.html
    
https://imbalanced-learn.org/stable/references/generated/imblearn.under_sampling.RandomUnderSampler.html

In [46]:
# Undersampling 
from imblearn.under_sampling import RandomUnderSampler

def undersample_variants(numeric_features):
    scaler = ColumnTransformer([("num", StandardScaler(), numeric_features)], remainder="drop")
    lr = Pipeline([("scale", scaler),
                   ("clf", LogisticRegression(solver="liblinear", C=1.0, max_iter=200,
                                              class_weight=None, random_state=random_state))])
    rf = Pipeline([("clf", RandomForestClassifier(n_estimators=200, n_jobs=-1,
                                                  class_weight=None, random_state=random_state))])
    return {"LR_Under": ("rus", lr), "RF_Under": ("rus", rf)}

def fit_eval_with_undersample(model_dict, X_train, y_train, X_test, y_test):
    rus = RandomUnderSampler(random_state=random_state)
    rows = []
    for name, (tag, pipe) in model_dict.items():
        X_tr_rs, y_tr_rs = rus.fit_resample(X_train, y_train)  # TRAIN ONLY
        pipe.fit(X_tr_rs, y_tr_rs)
        y_proba = pipe.predict_proba(X_test)[:,1] if hasattr(pipe[-1], "predict_proba") else pipe.predict(X_test)
        y_pred  = pipe.predict(X_test)
        rows.append({
            "model": name,
            "roc_auc": roc_auc_score(y_test, y_proba),
            "pr_auc":  average_precision_score(y_test, y_proba),
            "f1":      f1_score(y_test, y_pred, zero_division=0),
            "bacc":    balanced_accuracy_score(y_test, y_pred),
            **dict(zip(["tn","fp","fn","tp"], confusion_matrix(y_test, y_pred).ravel()))
        })
    return pd.DataFrame(rows).sort_values(["pr_auc","roc_auc"], ascending=False).reset_index(drop=True)


Two pragmatic options:

IsolationForest → returns anomaly scores; treat “anomalies” as predicted fraud.

One-Class SVM / LOF are alternatives; IsolationForest scales well.


Sources; 

https://scikit-learn.org/stable/modules/model_evaluation.html

In [47]:
def evaluate_isolation_forest(X_train, y_train, X_test, y_test, contam="auto"):
    # Train unsupervised on TRAIN features (no labels used)
    iso = IsolationForest(random_state=random_state, contamination=contam)
    iso.fit(X_train.drop(columns=[time_col], errors="ignore"))  # drop time col if present

    # Decision function -> higher = more normal; invert to get fraud score
    scores = iso.decision_function(X_test.drop(columns=[time_col], errors="ignore"))
    y_proba = (-scores - (-scores).min()) / ((-scores).max() - (-scores).min() + 1e-12)
    y_pred  = (y_proba >= 0.5).astype(int)  # fixed threshold for comparability

    return pd.DataFrame([{
        "model": "IsolationForest",
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc":  average_precision_score(y_test, y_proba),
        "f1":      f1_score(y_test, y_pred, zero_division=0),
        "bacc":    balanced_accuracy_score(y_test, y_pred),
        **dict(zip(["tn","fp","fn","tp"], confusion_matrix(y_test, y_pred).ravel()))
    }])


In [50]:
# Split temporally (80/20 here to reflect your function)
X_train, y_train, X_test, y_test = temporal_split(cc, time_col, class_col, frac=0.8)

# Features
features = feature_columns(cc, class_col)

# Baselines
base_df = evaluate_models(make_baseline_models(features), X_train, y_train, X_test, y_test)

# SMOTE
smote_df = fit_eval_with_smote(smote_variants(features), X_train, y_train, X_test, y_test)

# Under-sampling
under_df = fit_eval_with_undersample(undersample_variants(features), X_train, y_train, X_test, y_test)

# Anomaly (unsupervised)
anom_df = evaluate_isolation_forest(X_train, y_train, X_test, y_test)

# Compare (concatenate)
comparison = pd.concat([base_df, smote_df, under_df, anom_df], ignore_index=True)
comparison


,model,roc_auc,pr_auc,f1,bacc,tn,fp,fn,tp
0,RF_baseline,0.949403,0.783699,0.803150,0.839991,56886,1,24,51
1,LR_baseline,0.975349,0.739719,0.694215,0.779965,56883,4,33,42
2,RF_SMOTE,0.974728,0.807935,0.828571,0.886605,56880,7,17,58
3,LR_SMOTE,0.984313,0.799499,0.329897,0.924478,56638,249,11,64
4,RF_Under,0.988186,0.730649,0.087742,0.940967,55480,1407,7,68
5,LR_Under,0.983309,0.434655,0.109060,0.924087,55835,1052,10,65
6,IsolationForest,0.943443,0.029319,0.065523,0.635210,56342,545,54,21


Interpretation tips (what you should look for):

PR-AUC gains with SMOTE vs. baseline → better minority retrieval.

F1/Recall often increase with SMOTE/undersampling, but check FP inflation (precision drop).

IsolationForest may have lower PR-AUC but can surface rare signals—useful as a sanity reference.

Prefer the method that generalizes across both datasets with stable PR-AUC and balanced FN/FP trade-offs.
Refs: Fraud Detection Handbook (validation goals & metrics), imbalanced-learn user guide. 
Fraud Detection Handbook
+2
Fraud Detection Handbook
+2

Notes on reporting (to match your write-up)

State the temporal split and no resampling of test clearly (methodological backbone). 
Fraud Detection Handbook

Report PR-AUC, ROC-AUC, F1, Balanced Accuracy, plus the confusion matrix entries to show FP cost. 
Scikit-learn

For SMOTE/undersampling, name the exact library and version (e.g., imbalanced-learn 0.14.x). 
imbalanced-learn.org

If you later add hyperparameter tuning, use TimeSeriesSplit rather than random CV. 
Scikit-learn

Quick source list you can cite in the notebook

SMOTE / RandomUnderSampler (imbalanced-learn docs): API, usage, and notes on sampling strategy. 
imbalanced-learn.org
+2
imbalanced-learn.org
+2

Validation strategies for fraud (FDH): temporal validation & evaluation framing. 
Fraud Detection Handbook
+1

Metrics (sklearn): average precision (PR-AUC) & model evaluation overview. 
Scikit-learn
+1

Temporal CV (sklearn TimeSeriesSplit): use for any grid search/tuning on time-ordered data.